# SKA-like drift scan across the galactic plane (experiment 003)

Sidelobe stress test for the drift-scan strategy. Identical telescope, beams,
noise (same per-night seeds), and map-making recipe as `ska_drift_scan.ipynb`
(experiments 001/002) — the **only change is the LST window**: the same
az = 0°, el = 52°/50°/48° drift now starts at LST ≈ 280°, so the Dec ≈ +9°
strip sweeps galactic latitude **b = +7° → −6°**, crossing the plane
mid-window at l ≈ 43° (inner galaxy, several hundred K at 350 MHz).

Experiment 002 found that unmodelled −23 dB sidelobes cost essentially nothing
at b ≈ +50° (mismatch ≡ matched-gauss to ~3 mK, buried under ~2.4 K of 1/f
residual). Here the sidelobe rings sweep coherently through emission 10–100×
brighter, so that null result gets its proper test. Same three scenarios:

1. **gauss** — matched Gaussian forward/inverse (idealised control).
2. **airy** — matched tapered-aperture forward/inverse (modelled sidelobes).
3. **mismatch** — tapered-aperture sky through the Gaussian operator
   (*unmodelled* sidelobes, the realistic survey case). Shares the gauss
   operator, mask, prior, and noise realization — the mismatch-vs-gauss
   difference is purely the unmodelled sidelobe pickup.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'DSA')))

import pickle
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from astropy.coordinates import EarthLocation
from astropy import units as u

from limTOD import TODSim, HPW_mapmaking
from ska_common import (SKA_LAT, SKA_LON, SKA_HGT, ska_beam_func,
                        gdsm_equatorial_sky_model,
                        ska_airy_beam_func, ska_beam_fwhm_deg,
                        tapered_aperture_fwhm_deg,
                        drift_scan_night, azel_to_radec, save_results_pdf)
from dsa_vis import plot_map_compare


In [ ]:
# Identical configuration to ska_drift_scan.ipynb ...
freq_list = [350]  # MHz
BEAMS = {
    "gauss": dict(func=ska_beam_func,
                  fwhm=ska_beam_fwhm_deg(freq_list[0]),
                  label="Gaussian (1.22 λ/D)"),
    "airy": dict(func=ska_airy_beam_func,
                 fwhm=tapered_aperture_fwhm_deg(freq_list[0]),
                 label="Tapered aperture (−14 dB edge taper)"),
}
sky_Nside = 64
beam_Nside = 64
sky_Nside_map = 64
truncate_frac_thres = 1e-5
SEED = 0
WHITE_VAR = 2.5e-6

DRIFT_AZ = 0.0
ELEVATIONS = [52.0, 50.0, 48.0]
NIGHT_DURATION_S = 3600.0
DT = 2.0

# ... except the start time: LST ≈ 280° puts the plane crossing (l ≈ 43°,
# RA ≈ 288°, Dec ≈ +9°) in the middle of the 15°-wide drift window.
START_UTC = "2024-04-16 03:34:00"

site = EarthLocation(lat=SKA_LAT * u.deg, lon=SKA_LON * u.deg,
                     height=SKA_HGT * u.m)

nights = []
for night, el in enumerate(ELEVATIONS):
    tlist, azlist = drift_scan_night(NIGHT_DURATION_S, dt=DT,
                                     night=night, azimuth_deg=DRIFT_AZ)
    nights.append(dict(night=night, el=el, tlist=tlist, azlist=azlist))

# Boresight tracks over the GDSM patch: the plane crossing is obvious.
from astropy.coordinates import SkyCoord
fwhm_ref = BEAMS["airy"]["fwhm"]
fig, ax = plt.subplots(figsize=(8, 4))
for n in nights:
    ra, dec = azel_to_radec(DRIFT_AZ, n["el"], n["tlist"][::30],
                            START_UTC, site)
    b_gal = SkyCoord(ra=ra * u.deg, dec=dec * u.deg).galactic.b.deg
    line, = ax.plot(ra, dec, lw=1.5,
                    label=f"night {n['night']}: el = {n['el']:.0f}°, "
                          f"b = {b_gal[0]:+.0f}° → {b_gal[-1]:+.0f}°")
    ax.fill_between(ra, dec - fwhm_ref / 2, dec + fwhm_ref / 2,
                    alpha=0.15, color=line.get_color())
ax.set_xlabel("Right Ascension [deg]")
ax.set_ylabel("Declination [deg]")
ax.invert_xaxis()
ax.set_title("Galactic-plane drift tracks (bands = beam FWHM)")
ax.legend()
ax.grid(alpha=0.3)
os.makedirs("figures", exist_ok=True)
fig.savefig("figures/galplane_drift_tracks.png", dpi=200,
            bbox_inches="tight")
plt.show()


In [ ]:
# Simulate one TOD per night per beam (caches tagged _galplane).
tod_data = {}
for tag, b in BEAMS.items():
    cache = f"simulated_TODs_ska_drift_galplane_{tag}.npz"
    try:
        data = np.load(cache, allow_pickle=True)
        tod_data[tag] = dict(
            TOD_group=[np.asarray(t, np.float64) for t in data["TOD_group"]],
            LST_deg_list_group=[np.asarray(x, np.float64)
                                for x in data["LST_deg_list_group"]],
            azimuth_deg_list_group=[np.asarray(x, np.float64)
                                    for x in data["azimuth_deg_list_group"]],
            elevation_deg_list_group=[np.asarray(x, np.float64)
                                      for x in data["elevation_deg_list_group"]],
        )
        print(f"[{tag}] loaded {len(tod_data[tag]['TOD_group'])} cached "
              f"TODs from {cache}")
        continue
    except (FileNotFoundError, KeyError):
        pass

    tod_sim = TODSim(
        ant_latitude_deg=SKA_LAT,
        ant_longitude_deg=SKA_LON,
        ant_height_m=SKA_HGT,
        beam_func=b["func"],
        sky_func=gdsm_equatorial_sky_model,
        beam_nside=beam_Nside,
        sky_nside=sky_Nside,
    )
    groups = dict(TOD_group=[], LST_deg_list_group=[],
                  azimuth_deg_list_group=[], elevation_deg_list_group=[])
    print(f"[{tag}] generating simulated TODs (one per night)...")
    for n in nights:
        # Same seed per night across beams — identical noise realizations.
        np.random.seed(SEED + n["night"])
        tod_array, _, _, lst_deg = tod_sim.generate_TOD(
            freq_list=freq_list,
            time_list=n["tlist"],
            azimuth_deg_list=n["azlist"],
            elevation_deg=n["el"],
            start_time_utc=START_UTC,
            white_noise_var=WHITE_VAR,
            return_LSTs=True,
            normalize_beam=False,
            truncate_frac_thres=truncate_frac_thres,
        )
        groups["TOD_group"].append(np.asarray(tod_array[0], np.float64))
        groups["LST_deg_list_group"].append(np.asarray(lst_deg, np.float64))
        groups["azimuth_deg_list_group"].append(
            np.asarray(n["azlist"], np.float64))
        groups["elevation_deg_list_group"].append(
            n["el"] * np.ones_like(groups["TOD_group"][-1]))
    np.savez(cache, freq_list=freq_list, **groups)
    tod_data[tag] = groups
    print(f"[{tag}] done! Saved to {cache}")

# Night-0 TOD: the plane crossing dominates; the gauss/airy difference now
# shows the sidelobes loading in plane emission well before/after the main
# lobe crosses it. Log inset of the |difference| makes that visible.
fig, (ax, axd) = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                              height_ratios=[2, 1])
for tag, b in BEAMS.items():
    ax.plot(nights[0]["tlist"] / 60.0, tod_data[tag]["TOD_group"][0],
            lw=0.8, label=b["label"])
ax.set_ylabel("TOD [K]")
ax.legend()
ax.grid(alpha=0.3)
diff = (tod_data["airy"]["TOD_group"][0]
        - tod_data["gauss"]["TOD_group"][0])
axd.plot(nights[0]["tlist"] / 60.0, diff, lw=0.8, color="0.3")
axd.set_xlabel("Time into night 0 [min]")
axd.set_ylabel("airy − gauss [K]")
axd.grid(alpha=0.3)
fig.savefig("figures/galplane_tod_night0.png", dpi=200,
            bbox_inches="tight")
plt.show()


In [ ]:
# Operators per beam (cached, tagged _galplane).
mapmakers = {}
for tag, b in BEAMS.items():
    cache = f"mapmaker_ops_ska_drift_galplane_{tag}_ns{sky_Nside_map}.pkl"
    try:
        with open(cache, "rb") as f:
            mapmakers[tag] = pickle.load(f)
        print(f"[{tag}] loaded cached HPW_mapmaker operator from {cache}")
        continue
    except (FileNotFoundError, EOFError):
        pass
    beam_map = b["func"](freq=freq_list[0], nside=beam_Nside)
    mapmakers[tag] = HPW_mapmaking(
        beam_map=beam_map,
        LST_deg_list_group=tod_data[tag]["LST_deg_list_group"],
        lat_deg=SKA_LAT,
        azimuth_deg_list_group=tod_data[tag]["azimuth_deg_list_group"],
        elevation_deg_list_group=tod_data[tag]["elevation_deg_list_group"],
        threshold=0.05,
        nside_target=sky_Nside_map,
        beam_truncate_frac_thres=truncate_frac_thres,
    )
    with open(cache, "wb") as f:
        pickle.dump(mapmakers[tag], f)
    print(f"[{tag}] built HPW_mapmaker and cached to {cache}")

for tag, mm in mapmakers.items():
    print(f"[{tag}] pixel selection: {len(mm.pixel_indices)} pixels "
          f"at nside={sky_Nside_map}")


In [ ]:
# Map-making: same recipe and scenarios as ska_drift_scan.ipynb.
sky_truth_full = gdsm_equatorial_sky_model(freq=freq_list[0], nside=sky_Nside_map)


def solve_both(mm, TOD_group, assumed_fwhm_deg):
    '''Solve {no HP, HP 2 mHz} with the standard prior/noise recipe.'''
    pix = mm.pixel_indices
    sky_truth = sky_truth_full[pix]

    prior_mean = hp.smoothing(sky_truth_full,
                              fwhm=np.radians(assumed_fwhm_deg))[pix]
    prior_sigma_K = max(float(np.std(sky_truth)), 1e-3)
    prior_inv_cov = np.ones_like(sky_truth) / prior_sigma_K**2

    nv_floor = WHITE_VAR * (1e-3 * float(np.mean(sky_truth)))**2
    noise_variance = [WHITE_VAR * (np.asarray(op) @ sky_truth)**2 + nv_floor
                      for op in mm.Tsys_operators]

    common = dict(
        TOD_group=TOD_group,
        dtime=DT,
        Tsky_prior_mean=prior_mean,
        Tsky_prior_inv_cov_diag=prior_inv_cov,
        noise_variance=noise_variance,
        regularization=1e-12,
        return_full_cov=False,
    )
    est_noHP, _ = mm(**common)
    est_HP, _ = mm(
        **common,
        use_high_pass=True,
        cutoff_freq_group=np.array([2e-3] * len(TOD_group)),
        filter_order=2,
    )
    return {"no HP": est_noHP, "HP 2 mHz": est_HP}, sky_truth


SCENARIOS = {
    "gauss": (mapmakers["gauss"], tod_data["gauss"], BEAMS["gauss"]["fwhm"]),
    "airy": (mapmakers["airy"], tod_data["airy"], BEAMS["airy"]["fwhm"]),
    "mismatch": (mapmakers["gauss"], tod_data["airy"], BEAMS["gauss"]["fwhm"]),
}
SCENARIO_PIX = {tag: mm.pixel_indices for tag, (mm, _, _) in SCENARIOS.items()}

results = {}
truths = {}
for tag, (mm, data, fwhm) in SCENARIOS.items():
    solved, truths[tag] = solve_both(mm, data["TOD_group"], fwhm)
    for hp_label, est in solved.items():
        results[(tag, hp_label)] = est

rms_rows = []
print(f"{'scenario':10s} {'filter':10s} residual RMS")
for (tag, hp_label), est in results.items():
    rms = np.std(est - truths[tag]) * 1e3
    rms_rows.append((f"{tag} / {hp_label}", f"{rms:9.1f} mK"))
    print(f"{tag:10s} {hp_label:10s} {rms:10.1f} mK")

# The controlled sidelobe measurement: mismatch minus matched-gauss (same
# operator, mask, prior, noise) — any difference is unmodelled sidelobes.
for hp_label in ("no HP", "HP 2 mHz"):
    d = results[("mismatch", hp_label)] - results[("gauss", hp_label)]
    line = (f"unmodelled-sidelobe map error ({hp_label}): "
            f"RMS = {np.std(d)*1e3:9.1f} mK, "
            f"max |Δ| = {np.max(np.abs(d))*1e3:9.1f} mK")
    rms_rows.append((f"mismatch − gauss / {hp_label}",
                     f"{np.std(d)*1e3:9.1f} mK"))
    print(line)


In [ ]:
SCENARIO_LABELS = {
    "gauss": BEAMS["gauss"]["label"],
    "airy": BEAMS["airy"]["label"],
    "mismatch": "unmodelled sidelobes (airy sky, Gaussian operator)",
}
for tag, label in SCENARIO_LABELS.items():
    for hp_label, suffix in [("no HP", "noHP"), ("HP 2 mHz", "HP")]:
        fig = plot_map_compare(
            sky_est=results[(tag, hp_label)],
            sky_truth=truths[tag],
            nside=sky_Nside_map,
            pixel_indices=SCENARIO_PIX[tag],
            freq_mhz=freq_list[0],
            strategy_name=f"Galactic plane, {label}, {hp_label}",
            savepath=f"figures/map_ska_galplane_{tag}_{suffix}.png",
        )
        plt.show()


In [ ]:
save_results_pdf(
    3, "galplane-drift-sidelobe-stress",
    "Galactic-plane sidelobe stress test: identical drift design, beams, "
    "noise seeds and map-making recipe as experiments 001/002, but the LST "
    f"window (start {START_UTC} UTC, LST ≈ 280°) sweeps the Dec ≈ +9° strip "
    "across the galactic plane (b = +7° → −6°, crossing at l ≈ 43°). "
    "Scenarios: matched Gaussian, matched tapered aperture, and beam "
    "mismatch (tapered-aperture sky through the Gaussian operator). The "
    "mismatch − gauss rows isolate the unmodelled-sidelobe map error "
    "(same operator, mask, prior and noise realization).",
    ["figures/galplane_drift_tracks.png", "figures/galplane_tod_night0.png"]
    + [f"figures/map_ska_galplane_{t}_{s}.png"
       for t in ("gauss", "airy", "mismatch") for s in ("noHP", "HP")],
    rms_table=rms_rows,
)
